# Transforming Data with Pandas

In the previous notebook, you learned how to load, inspect, select, filter, and sort data with pandas. Now we'll move on to **transforming** data — grouping rows to compute summaries and creating new columns from existing ones.

In this notebook, you'll learn how to:
- Group data and compute aggregate statistics with `.groupby()`
- Apply multiple summary functions at once with `.agg()`
- Create, rename, and drop columns

We'll continue working with the [Gapminder](https://www.gapminder.org/) dataset from yesterday.

In [ ]:
import pandas as pd

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminder_unfiltered.csv"
df = pd.read_csv(url)
df.head()

---
## 1. Grouping and Aggregation

One of the most powerful patterns in data analysis is **split-apply-combine**:
1. **Split** the data into groups (e.g., by continent)
2. **Apply** a function to each group (e.g., compute the mean)
3. **Combine** the results back into a table

In pandas, this is done with `.groupby()`.

In [ ]:
# Average life expectancy by continent
df.groupby("continent")["lifeExp"].mean()

In [ ]:
# Multiple statistics at once with .describe()
df.groupby("continent")["lifeExp"].describe()

### Grouping by multiple columns

You can group by more than one column to get finer breakdowns.

In [ ]:
# Average life expectancy by continent and year
by_continent_year = df.groupby(["continent", "year"])["lifeExp"].mean()
by_continent_year.head(15)

### Applying multiple functions with `.agg()`

`.agg()` lets you apply several different functions at once.

**🤔 Guess first:** The next cell runs `.agg(["min", "mean", "max"])` on life expectancy. Before running it, predict the **shape** of the result — how many rows, how many columns?


In [ ]:
# For each continent: min, mean, and max life expectancy
df.groupby("continent")["lifeExp"].agg(["min", "mean", "max"])

### Combining groupby with filtering

A very common pattern: filter first, then group.

**🤔 Guess first:** The next cell filters to 2007 and then groups by continent. Before running it, predict: how many **rows** will the output have? Commit to a number.


In [ ]:
# Average GDP per capita by continent, but only for the year 2007
# Pro Tip: We use .copy() to create a fully independent dataframe, preventing future warnings!
data_2007 = df[df["year"] == 2007].copy()

data_2007.groupby("continent")["gdpPercap"].mean().sort_values(ascending=False)

### Exercise 1: Grouping

**1a.** What is the total population of each continent in 2007?

*Hint: filter to 2007, then `.groupby("continent")["pop"].sum()`*

In [ ]:
# Your code here


**1b.** For each continent, what is the **median** GDP per capita in 2007? Sort the result from highest to lowest.

*Hint: use `.median()` instead of `.mean()`.*

In [ ]:
# Your code here


**1c (stretch).** Which country had the highest life expectancy in **each** continent in 2007?

*Hint: filter to 2007 first and save it as a variable (like `data_2007`), then try `.loc[data_2007.groupby("continent")["lifeExp"].idxmax()]` — `idxmax()` returns the row index of the maximum value in each group.*

In [ ]:
# Your code here


---
## 2. Creating and Modifying Columns

You can create new columns from existing ones using arithmetic or string operations. This is useful for derived quantities.

**🤔 Guess first:** `df` currently has **6 columns**. The next two cells each add one new column. Before running them, predict what `df.shape` will be afterwards.


In [ ]:
# Total GDP = GDP per capita * population
df["gdpTotal"] = df["gdpPercap"] * df["pop"]
df[["country", "year", "gdpPercap", "pop", "gdpTotal"]].head()

In [ ]:
# Population in millions (easier to read)
df["pop_millions"] = df["pop"] / 1_000_000
df[["country", "year", "pop", "pop_millions"]].head()

### Renaming columns

Use `.rename()` to give columns clearer names.

**🤔 Guess first:** The comment in the next cell says `.rename()` returns a **new** DataFrame. So after the cell runs — will `df` itself have the new column names, or the old ones? How would you check?


In [ ]:
# Rename for clarity (this returns a new DataFrame; original is unchanged)
renamed = df.rename(columns={"lifeExp": "life_expectancy", "gdpPercap": "gdp_per_capita"})
renamed.columns.tolist()

### Dropping columns

Use `.drop()` to remove columns you no longer need.

In [ ]:
# Remove the columns we created
df = df.drop(columns=["gdpTotal", "pop_millions"])
df.columns.tolist()

> **Out-of-order execution in action:** The cell above just dropped `gdpTotal` and `pop_millions` from `df`. If you scroll back up and re-run the cells that *created* those columns, they'll reappear. Then if you re-run the drop cell, they'll disappear again. This is why notebooks can feel unpredictable — the state of `df` depends on which cells you ran and in what order.
>
> When you're confused about what a variable contains, just inspect it:

In [ ]:
# Run this cell anytime to check what df looks like right now
print("Columns:", list(df.columns))
print("Shape:", df.shape)

### Exercise 2: New columns

**2a.** Create a new column called `lifeExp_months` that converts life expectancy from years to months.

Display the first few rows of `country`, `year`, `lifeExp`, and your new column.

In [ ]:
# Your code here


**2b (stretch).** For the year 2007 only, create a column called `pop_rank` that ranks countries by population (largest = 1).

*Hint: When filtering data to create a subset (like just the year 2007), and then adding new columns to that subset, always use `.copy()` to avoid pandas' infamous `SettingWithCopyWarning`. For example: `data_2007 = df[df["year"] == 2007].copy()`. Then try creating your new column using `.rank(ascending=False)` on the `pop` column!*

In [ ]:
# Your code here


---
## 🏠 Optional Homework (after hours)

Optional exercises for tonight — they only use what's in this notebook. Bring questions tomorrow!

### Homework 1: World population over time

Group the full dataset by `year` and compute the **total world population** for each year. Print the result. Roughly how much did the world population grow between the first and last year in the data?


In [ ]:
# Homework 1 — your code here


### Homework 2: Top 5 economies

Recreate the `gdpTotal` column (`gdpPercap * pop`). Then, for the year **2007 only**, find the 5 countries with the largest total GDP.

*Hint: new column → filter to 2007 → `.sort_values(ascending=False)` → `.head(5)`.*


In [ ]:
# Homework 2 — your code here


### Homework 3 (stretch): The life-expectancy gap

For each continent in 2007, compute the **gap** between the highest and lowest life expectancy using a single `.agg(["min", "max"])`, then subtract the two columns to make a `gap` column. Which continent has the widest gap?

*Hint: the result of `.agg()` is a DataFrame — `result["max"] - result["min"]` works.*


In [ ]:
# Homework 3 — your code here


---
## What's next?

You now have a complete toolkit for working with tabular data in pandas — from loading and inspecting, to filtering and sorting, to grouping and creating new columns. Next up: **Loading Data** — getting data into pandas from URLs, libraries, local files, and whole folders of files. After that, we'll tackle cleaning messy real-world data.

## Key Points

- **Group and aggregate** with `.groupby()` — the split-apply-combine pattern.
- Use `.agg()` to apply multiple summary functions (min, mean, max, etc.) at once.
- **Combine filtering with groupby** for targeted summaries (e.g., only 2007 data).
- **Create columns** with `df["new"] = ...`. Rename with `.rename()`, drop with `.drop()`.